In [29]:
import pandas as pd
import numpy as np
import joblib

## Load Models

In [30]:
binary_model = joblib.load("tuned_xgboost_model.pkl")
carc_model = joblib.load("carc_prediction_model.pkl")

print("Models loaded successfully.")

Models loaded successfully.


## LLoad encoders

In [31]:
target_encoder = joblib.load("target_encoder.pkl")
carc_encoder = joblib.load("carc_label_encoder.pkl")

print("Target encoder classes:", target_encoder.classes_)
print("CARC encoder classes:", carc_encoder.classes_)

Target encoder classes: ['Approved' 'Denied']
CARC encoder classes: ['CO-109' 'CO-16' 'CO-18' 'CO-197' 'CO-29' 'CO-45' 'CO-50' 'CO-97']


In [32]:
# This is the fix from before: load the actual test split used to
# report metrics in Notebook 10, not the full unsplit dataframe.
X_test = joblib.load("X_test.pkl")
y_test = joblib.load("y_test.pkl")

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (3000, 23)
y_test shape: (3000,)


In [33]:
# Used later only to pull a human-readable example claim to demo
# predict_claim() on. NOT used for accuracy validation - X_test is.
df = pd.read_csv("claims_preprocessed_v3.csv")
print(df.shape)

(15000, 30)


## Prediction Function

In [34]:
def predict_claim(claim_row: pd.DataFrame) -> dict:
    """
    claim_row: a single-row DataFrame containing ONLY the 23 model
    input features (no Claim_Status/CARC_Code/etc columns).
    """
    status_pred = binary_model.predict(claim_row)[0]
    status_prob = binary_model.predict_proba(claim_row)[0]
    claim_status = target_encoder.inverse_transform([status_pred])[0]

    result = {
        "Claim_Status": claim_status,
        "Claim_Confidence": float(status_prob.max()),
    }

    if claim_status == "Denied":
        carc_pred = carc_model.predict(claim_row)[0]
        carc_prob = carc_model.predict_proba(claim_row)[0]
        result["CARC_Code"] = carc_encoder.inverse_transform([carc_pred])[0]
        result["CARC_Confidence"] = float(carc_prob.max())
    else:
        result["CARC_Code"] = None
        result["CARC_Confidence"] = None

    return result

print("predict_claim() defined.")

predict_claim() defined.


## Demo: Predict One Example Claim (illustration only — not a metric)

In [35]:
drop_cols = ["Claim_Status", "CARC_Code", "Denial_Reason",
             "Procedure_Desc", "Place_of_Service_Desc",
             "Service_Date", "Claim_Submission_Date"]

example_claim = df.iloc[[0]].drop(columns=drop_cols, errors="ignore")
result = predict_claim(example_claim)

print("Predicted:", result)
print("Actual Claim_Status:", df.iloc[0]["Claim_Status"])

Predicted: {'Claim_Status': 'Denied', 'Claim_Confidence': 0.5187186598777771, 'CARC_Code': 'CO-50', 'CARC_Confidence': 0.8021241426467896}
Actual Claim_Status: Approved


In [36]:
X_test_carc = joblib.load("X_test_carc.pkl")
y_test_carc = joblib.load("y_test_carc.pkl")

print("X_test_carc shape:", X_test_carc.shape)
print("y_test_carc shape:", y_test_carc.shape)

X_test_carc shape: (542, 23)
y_test_carc shape: (542,)


## Real Validation: Accuracy on the Held-Out Test Set

In [38]:
correct_carc = 0
total_carc = len(X_test_carc)

for row_position in range(total_carc):
    test_row = X_test_carc.iloc[[row_position]]

    carc_pred = carc_model.predict(test_row)[0]
    predicted_carc = carc_encoder.inverse_transform([carc_pred])[0]
    actual_carc = carc_encoder.inverse_transform([y_test_carc.iloc[row_position]])[0]

    if predicted_carc == actual_carc:
        correct_carc += 1

print(f"Correct CARC Predictions: {correct_carc}/{total_carc}")
print(f"CARC Accuracy on held-out test set: {correct_carc/total_carc:.2%}")

Correct CARC Predictions: 372/542
CARC Accuracy on held-out test set: 68.63%


# FINAL VERIFICATION: Both models, real held-out test sets


In [39]:
# =====================================================
# FINAL VERIFICATION: Both models, real held-out test sets
# =====================================================

# ---- Model 1: Denial Prediction ----
correct = 0
total = len(X_test)

for row_position in range(total):
    test_row = X_test.iloc[[row_position]]
    status_pred = int(binary_model.predict(test_row)[0])
    predicted_status = "Denied" if status_pred == 1 else "Approved"
    actual_status = "Denied" if y_test[row_position] == 1 else "Approved"
    if predicted_status == actual_status:
        correct += 1

print("MODEL 1 - Denial Prediction")
print(f"Correct Predictions: {correct}/{total}")
print(f"Accuracy on REAL held-out test set: {correct/total:.2%}")
print()

# ---- Model 2: CARC Reason Prediction ----
correct_carc = 0
total_carc = len(X_test_carc)

for row_position in range(total_carc):
    test_row = X_test_carc.iloc[[row_position]]
    carc_pred = carc_model.predict(test_row)[0]
    predicted_carc = carc_encoder.inverse_transform([carc_pred])[0]
    actual_carc = carc_encoder.inverse_transform([y_test_carc.iloc[row_position]])[0]
    if predicted_carc == actual_carc:
        correct_carc += 1

print("MODEL 2 - CARC Reason Prediction")
print(f"Correct CARC Predictions: {correct_carc}/{total_carc}")
print(f"CARC Accuracy on held-out test set: {correct_carc/total_carc:.2%}")

MODEL 1 - Denial Prediction
Correct Predictions: 2039/3000
Accuracy on REAL held-out test set: 67.97%

MODEL 2 - CARC Reason Prediction
Correct CARC Predictions: 372/542
CARC Accuracy on held-out test set: 68.63%
